In [ ]:

import pandas as pd
import zipfile
import urllib.request

url = "https://archive.ics.uci.edu/static/public/320/student+performance.zip"

urllib.request.urlretrieve(url, "student.zip")

with zipfile.ZipFile("student.zip", "r") as z:
    z.extractall("student_data")

df = pd.read_csv(
    "student_data/student-mat.csv",
    sep=";"
)

print("Dataset loaded successfully!")
print("Rows and columns:", df.shape)

display(df.head())


# Assignment No. 4: Machine Learning Model Application & Evaluation

**Course:** Introduction to Data Science

**Title:** Apply and Compare Machine Learning Models on a Real-World Dataset

### 1. Problem Definition

This project uses the UCI Student Performance Dataset to predict whether a student is at risk of poor academic performance. Students with a final mathematics grade below 10 out of 20 are classified as At Risk (1), while students with a grade of 10 or above are classified as Not At Risk (0).

**Problem Type:** Binary Classification

**Features (X):** Student age, study time, absences, previous failures, family support, school support, and other available demographic and behavioral features.

**Target (y):** At_Risk

**Objective:** Train and compare four machine learning models and identify the most suitable model using evaluation metrics.

In [ ]:

import pandas as pd
import zipfile
import urllib.request

url = "https://archive.ics.uci.edu/static/public/320/student+performance.zip"

urllib.request.urlretrieve(url, "student.zip")

with zipfile.ZipFile("student.zip", "r") as z:
    z.extractall("student_data")

df = pd.read_csv(
    "student_data/student-mat.csv",
    sep=";"
)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
display(df.head())


In [ ]:

print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nData types:")
print(df.dtypes)


The dataset contains student academic and personal information.



The code checks missing values, duplicate rows and data types.



We will check and remove duplicate rows before training the models.



Categorical features will be encoded into numerical form by the preprocessing pipeline.

In [ ]:

df = df.drop_duplicates().copy()

df["At_Risk"] = (df["G3"] < 10).astype(int)

print("Target distribution:")
print(df["At_Risk"].value_counts())

print("\nTarget labels:")
print("1 = At Risk")
print("0 = Not At Risk")


In [ ]:

X = df.drop(columns=["G1", "G2", "G3", "At_Risk"])
y = df["At_Risk"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

display(X.head())


In [ ]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))


In [ ]:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

numeric_cols = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_cols = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"),
         categorical_cols)
    ]
)

print("Numerical features:", len(numeric_cols))
print("Categorical features:", len(categorical_cols))


In [ ]:

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000, class_weight="balanced"
    ),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=5, class_weight="balanced",
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, max_depth=5,
        class_weight="balanced", random_state=42
    )
}

trained_models = {}

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessing", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)
    trained_models[name] = pipeline

    print(name, "trained successfully!")


In [ ]:

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

results = []

for name, model in trained_models.items():
    y_pred = model.predict(X_test)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(
            y_test, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_test, y_pred, zero_division=0
        ),
        "F1-Score": f1_score(
            y_test, y_pred, zero_division=0
        )
    })

    print("\nModel:", name)
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(
        y_test, y_pred, zero_division=0
    ))

results_df = pd.DataFrame(results)
display(results_df.round(3))


In [ ]:

comparison = results_df.sort_values(
    by="F1-Score", ascending=False
).reset_index(drop=True)

display(comparison.round(3))

print("Best model according to F1-score:")
print(comparison.iloc[0]["Model"])


In [ ]:

from sklearn.metrics import f1_score

fit_check = []

for name, model in trained_models.items():
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_f1 = f1_score(
        y_train, train_pred, zero_division=0
    )
    test_f1 = f1_score(
        y_test, test_pred, zero_division=0
    )

    fit_check.append({
        "Model": name,
        "Training F1": train_f1,
        "Testing F1": test_f1,
        "F1 Difference": train_f1 - test_f1
    })

display(pd.DataFrame(fit_check).round(3))


###  Final Conclusion

In this project, I applied four machine learning models: Logistic Regression, K-Nearest Neighbors, Decision Tree, and Random Forest. The UCI Student Performance Dataset was used to predict whether a student is at risk of poor academic performance.

The dataset was divided into training and testing sets. The models were evaluated using accuracy, precision, recall, F1-score, and confusion matrices.

The best-performing model according to F1-score was **[write the model name from your results]**, with an F1-score of **[write your actual value]** and a recall of **[write your actual value]**.

Recall was considered important because identifying students who may need academic support is a priority. The training and testing results were also compared to check for possible overfitting or underfitting.

In conclusion, this project demonstrates how machine learning models can be compared using appropriate evaluation metrics. The predictions are experimental and should not be used alone to make decisions about individual students.